# 서울시 공영주차장 "30분 뒤 혼잡도" 예측 — SVM 계열 모델 비교

혼잡도 3단계(여유 / 보통 / 혼잡)를 **점유율 = 현재 주차 대수 ÷ 총 주차면**으로 정의하고, 30분 뒤의 혼잡도를 예측합니다.

**비교 모델**

1. 기준선 A: 최빈 클래스 (학습 구간에서 가장 흔한 클래스)
2. 기준선 B: "지금 그대로" (현재 혼잡도를 30분 뒤에도 유지)
3. LinearSVC
4. RBF SVC
5. SVR로 30분 뒤 점유율을 예측한 뒤 같은 구간 기준으로 나눔

**평가**: 시간 순서대로 나눈 학습/검증/테스트, macro F1, 클래스별 재현율, 혼동 행렬

## 사용법

1. 바로 아래 **설정 셀**만 바꾸고, 위에서 아래로 순서대로 실행합니다(Run All).
2. 기본값은 `DEMO = True`입니다. 합성 데이터로 코드가 도는지만 확인하며, 결과는 **실제 성능이 아닙니다.**
3. 실제 데이터를 쓰려면 `DEMO = False`로 바꾸고 `DATA`에 경로를 적습니다. 두 가지 형식을 모두 읽습니다.
   - (a) 팀 저장소 수집기가 쌓는 `data/collected/raw/*.csv` (영문 헤더, `collected_at`에 `+09:00` 포함). 폴더로 `data/collected`를 주면 `raw/`만 읽고 `static/`은 건너뜁니다.
   - (b) 포털에서 내려받은 한글 헤더 CSV(cp949)를 여러 시점 모아 둔 것. 필수 컬럼은 `주차장코드`, `총 주차면`, `현재 주차 차량수`이고, 시각은 `collected_at`(직접 기록한 수집 시각, 권장) 또는 `현재 주차 차량수 업데이트시간`을 씁니다.
4. 스냅샷 CSV 한 장뿐이면 **분포 확인(EDA) 셀까지만** 실행하세요. 30분 뒤 예측은 시계열 수집 후에 가능합니다.
5. Colab이면 CSV를 먼저 업로드한 뒤 그 경로를 `DATA`에 적습니다.

## 0. 설정

In [ ]:
# ===================== 설정: 여기만 바꾸면 됩니다 =====================
DEMO = True                 # True: 합성 데이터로 코드만 점검(실제 성능 아님) / False: 아래 DATA 의 실제 데이터 사용
DATA = ["data/collected"]   # DEMO=False 일 때 읽을 CSV 파일 또는 CSV 가 든 폴더(여러 개 가능)
                            #   예) ["snapshot.csv"]   ["data/collected"]   ["portal_csvs/"]
OUTDIR = "results"          # 결과 저장 폴더

THR = (0.50, 0.85)          # 혼잡도 경계(점유율 비율, 퍼센트 아님): <0.50 여유, <0.85 보통, 그 이상 혼잡
INTERVAL_MIN = 10           # 수집 간격(분). 예측 시점(HORIZON_MIN)의 약수여야 함 (30분이면 5, 10, 15, 30)
HORIZON_MIN = 30            # 몇 분 뒤를 예측할지
TEST_FRAC = 0.2             # 시간 순서상 마지막 20% 를 테스트로
VAL_FRAC = 0.2              # 학습 구간의 마지막 20% 를 검증으로
MAX_TRAIN = 20000           # SVM 학습 표본 상한(0이면 제한 없음)
MAX_VAL = 10000             # 검증 표본 상한
NO_BALANCE = False          # True 면 클래스 가중치 균형(balanced)을 끈다
HOLIDAYS = []               # 공휴일 날짜. 예) ["2026-12-25"]
DROP_CONSTANT_LOTS = False  # True 면 관측 내내 차량 수가 변하지 않은 주차장(센서 정지 의심)을 제외
FAST = True                 # True: 하이퍼파라미터 격자를 줄여 빠르게(점검용). 최종 결과를 낼 때는 False
SEED = 42

## 1. 준비: 라이브러리, 상수

In [ ]:
from __future__ import annotations

import glob
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import confusion_matrix, f1_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR, LinearSVC

try:
    from IPython.display import display
except ImportError:  # 주피터가 아닌 환경
    display = print

warnings.filterwarnings("ignore", category=ConvergenceWarning)

CLASS_NAMES = ["여유", "보통", "혼잡"]
# 컬럼 이름 별칭: 포털에서 내려받은 한글 헤더 CSV와, 팀 수집기(src/collect.py)가 쌓는 영문 헤더 CSV를 모두 읽는다
COLUMN_ALIASES = {
    "lot_id": ["주차장코드", "PKLT_CD"],
    "lot_name": ["주차장명", "PKLT_NM"],
    "capacity": ["총 주차면", "TPKCT"],
    "current": ["현재 주차 차량수", "NOW_PRK_VHCL_CNT"],
    "updated_at": ["현재 주차 차량수 업데이트시간", "NOW_PRK_VHCL_UPDT_TM"],
    "provided": ["주차현황 정보 제공여부", "PRK_STTS_YN"],  # 1이면 20분 이내 실시간 연계, 0이면 미연계
    "collected_at": ["collected_at"],
}

# 차트 색(데이터 시각화 기본 팔레트의 파랑 계열 + 중립 잉크)
INK, INK2, GRID, AXIS, BLUE = "#191919", "#666666", "#e7e7e7", "#c2c2c2", "#257de0"
BLUE_SEQ = ["#cce3fd", "#84b9f3", "#338ced", "#1b64be", "#0e3d7e"]

## 2. 데이터 읽기

In [ ]:
def read_csv_any(path: str) -> pd.DataFrame:
    for enc in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(path, encoding=enc, dtype=str)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩을 판별하지 못했습니다: {path}")


def to_naive_kst(s: pd.Series) -> pd.Series:
    """시각 문자열 -> 한국 시간 기준, 시간대 없는 datetime.
    '2026-11-25T23:03:11+09:00'처럼 오프셋이 있으면 KST로 맞춘 뒤 오프셋을 떼고, 없으면 그대로 KST로 본다."""
    s = s.astype("string").str.strip()
    has_off = s.str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", regex=True, na=False)
    with_off = pd.to_datetime(s.where(has_off), errors="coerce", utc=True).dt.tz_convert("Asia/Seoul").dt.tz_localize(None)
    no_off = pd.to_datetime(s.where(~has_off), errors="coerce")
    return with_off.fillna(no_off)


def load_snapshots(paths: list[str], interval_min: int, drop_constant: bool = False) -> pd.DataFrame:
    files: list[str] = []
    for p in paths:
        if not os.path.exists(p):
            raise FileNotFoundError(
                f"경로가 없습니다: {p}\n       팀 저장소라면 먼저 'bash scripts/sync_data.sh' 로 data 브랜치를 받아야 합니다. "
                "data 브랜치가 아직 없다면 GitHub Actions 수집이 시작되지 않은 것입니다.")
        if os.path.isdir(p):
            # 팀 저장소의 data/collected 를 주면 raw/ 만 읽는다(static/ 은 주차장 정보라 제외)
            base = os.path.join(p, "raw") if os.path.isdir(os.path.join(p, "raw")) else p
            files += sorted(glob.glob(os.path.join(base, "*.csv")))
        else:
            files.append(p)
    if not files:
        raise FileNotFoundError("읽을 CSV 파일이 없습니다. raw/*.csv 가 실제로 쌓였는지 확인하세요.")

    inv = {alias: key for key, aliases in COLUMN_ALIASES.items() for alias in aliases}
    frames = []
    for f in files:
        raw = read_csv_any(f)
        raw.columns = [str(c).strip() for c in raw.columns]  # 헤더 앞뒤 공백 제거
        raw = raw.rename(columns={c: inv[c] for c in raw.columns if c in inv})
        need = [k for k in ("lot_id", "capacity", "current") if k not in raw.columns]
        if need:
            raise ValueError(f"{f}: 필수 컬럼 누락 {[' / '.join(COLUMN_ALIASES[k]) for k in need]}")
        ts_col = "collected_at" if "collected_at" in raw.columns else "updated_at"
        if ts_col not in raw.columns:
            raise ValueError(f"{f}: 시각 컬럼(collected_at 또는 업데이트시간)이 없습니다.")
        raw["ts"] = to_naive_kst(raw[ts_col])
        if ts_col == "collected_at" and "updated_at" in raw.columns:
            # 값이 얼마나 묵었는지(분). 수집 시각을 관측 시각으로 쓰므로 '30분 뒤'에는 이만큼의 오차가 섞인다
            raw["stale_min"] = (raw["ts"] - to_naive_kst(raw["updated_at"])).dt.total_seconds() / 60
        raw["source"] = os.path.basename(f)
        frames.append(raw)
    df = pd.concat(frames, ignore_index=True)

    df["capacity"] = pd.to_numeric(df["capacity"], errors="coerce")
    df["cur"] = pd.to_numeric(df["current"], errors="coerce")
    n0 = len(df)
    keep = df["ts"].notna() & df["capacity"].gt(0) & df["cur"].notna()
    if "provided" in df.columns:
        keep &= df["provided"].astype(str).str.strip().eq("1")  # 실시간 미연계 주차장 제외
    df = df[keep].copy()
    print(f"[load] 파일 {len(files)}개, 행 {n0:,} -> 유효 {len(df):,} (미연계·결측·면수0 제외)")
    if "stale_min" in df.columns and df["stale_min"].notna().any():
        s = df["stale_min"].dropna().clip(lower=0)
        print(f"[load] 값의 묵은 정도(수집 시각 - 업데이트 시각): 중앙값 {s.median():.0f}분, 90% {s.quantile(.9):.0f}분, "
              f"20분 초과 {(s > 20).mean() * 100:.1f}%")

    if "lot_name" not in df.columns:
        df["lot_name"] = df["lot_id"]
    df["occ"] = (df["cur"] / df["capacity"]).clip(0, 1.5)
    df["t"] = df["ts"].dt.floor(f"{interval_min}min")
    df = df.sort_values("ts").drop_duplicates(["lot_id", "t"], keep="last")

    # 격자 채움률: cron 누락·지연·미연계로 빠진 칸이 많으면 학습 행이 크게 줄어든다
    span = df.groupby("lot_id")["t"].agg(["min", "max", "count"])
    expect = ((span["max"] - span["min"]) / pd.Timedelta(minutes=interval_min)).round() + 1
    print(f"[load] {interval_min}분 격자 채움률 {span['count'].sum() / expect.sum() * 100:.1f}% "
          "(빠진 칸은 30분 뒤·과거 값이 없는 행으로 학습에서 빠집니다)")

    # 관측 내내 차량 수가 한 번도 안 바뀐 주차장(센서 정지 의심)은 '지금 그대로' 기준선을 부풀린다
    nun = df.groupby("lot_id")["cur"].agg(["nunique", "size"])
    const = nun.index[(nun["nunique"] == 1) & (nun["size"] >= 50)]
    if len(const):
        if drop_constant:
            df = df[~df["lot_id"].isin(const)]
            print(f"[load] 차량 수가 변하지 않은 주차장 {len(const)}개를 제외했습니다.")
        else:
            print(f"[경고] 관측 내내 차량 수가 변하지 않은 주차장이 {len(const)}개 있습니다(센서 정지 의심). "
                  "'지금 그대로' 기준선이 부풀 수 있으니 DROP_CONSTANT_LOTS = True 로 제외한 결과도 비교하세요.")

    # 수집 간격이 설정과 크게 다르면 경고
    gap = df.sort_values("t").groupby("lot_id")["t"].diff().dropna()
    if len(gap):
        med = gap.median() / pd.Timedelta(minutes=1)
        if med > 1.5 * interval_min:
            print(f"[경고] 주차장별 관측 간격 중앙값이 {med:.0f}분입니다. INTERVAL_MIN({interval_min})을 확인하세요.")
    return df[["lot_id", "lot_name", "t", "capacity", "cur", "occ"]].reset_index(drop=True)


def make_demo(n_lots: int = 60, days: int = 21, interval_min: int = 10, seed: int = 0) -> pd.DataFrame:
    """코드 검증용 합성 데이터. 실제 주차장 데이터가 아니다."""
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2026-11-02", periods=days * 24 * 60 // interval_min, freq=f"{interval_min}min")
    h = idx.hour.to_numpy() + idx.minute.to_numpy() / 60
    weekend = (idx.dayofweek >= 5).astype(float)
    rows = []
    for i in range(n_lots):
        cap = int(np.clip(rng.lognormal(3.7, 1.0), 10, 1200))
        base, amp = rng.uniform(0.2, 0.7), rng.uniform(0.1, 0.35)
        peak, wk = rng.uniform(11, 19), rng.uniform(-0.2, 0.2)
        e = np.zeros(len(idx))
        for k in range(1, len(idx)):
            e[k] = 0.9 * e[k - 1] + rng.normal(0, 0.03)
        occ = np.clip(base + amp * np.cos(2 * np.pi * (h - peak) / 24) + wk * weekend + e, 0, 1)
        rows.append(pd.DataFrame({"lot_id": f"D{i:03d}", "lot_name": f"demo-{i}", "t": idx,
                                  "capacity": cap, "cur": np.round(occ * cap), "occ": np.round(occ * cap) / cap}))
    return pd.concat(rows, ignore_index=True)

## 3. 혼잡도 구간, EDA

In [ ]:
def to_class(occ, thr) -> np.ndarray:
    """점유율 -> 0(여유) / 1(보통) / 2(혼잡). thr=(a, b): occ<a 여유, a<=occ<b 보통, occ>=b 혼잡."""
    return np.digitize(np.asarray(occ, dtype=float), bins=list(thr))


def share_table(occ, thr) -> list[float]:
    c = to_class(occ, thr)
    return [float((c == k).mean()) for k in range(3)]


def threshold_candidate_lines(occ) -> list[str]:
    """경계 후보별 클래스 비율(%). occ는 점유율 배열(어느 구간의 값을 넣느냐에 따라 '무엇을 보고 정했나'가 달라진다)."""
    occ = pd.Series(np.asarray(occ, dtype=float))
    t1, t2 = occ.quantile([1 / 3, 2 / 3])
    out = []
    for name, th in [("50/85 (기본)", (0.5, 0.85)), ("40/80", (0.4, 0.8)), ("60/90", (0.6, 0.9)),
                     (f"3분위 {t1:.2f}/{t2:.2f}", (t1, t2))]:
        s = share_table(occ, th)
        out.append(f"- {name}: " + " / ".join(f"{x * 100:.1f}" for x in s))
    return out


def eda(df: pd.DataFrame, thr, outdir: str) -> None:
    n_t, n_lot = df["t"].nunique(), df["lot_id"].nunique()
    span = df["t"].max() - df["t"].min()
    q = df["occ"].quantile([.1, .25, .5, .75, .9]).round(3).to_dict()
    lines = [
        "# EDA 요약",
        f"- 유효 관측 {len(df):,}건, 주차장 {n_lot}개, 고유 시점 {n_t}개",
        f"- 기간 {df['t'].min()} ~ {df['t'].max()} (폭 {span})",
        f"- 점유율 평균 {df['occ'].mean():.3f}, 중앙값 {df['occ'].median():.3f}, "
        f"면수 가중 평균 {df['cur'].sum() / df['capacity'].sum():.3f}",
        f"- 분위수(10/25/50/75/90%) {list(q.values())}",
        "",
        f"## 혼잡도 분포 (기준 {thr[0]:.2f} / {thr[1]:.2f})",
    ]
    sh = share_table(df["occ"], thr)
    cnt = np.bincount(to_class(df["occ"], thr), minlength=3)
    for k in range(3):
        lines.append(f"- {CLASS_NAMES[k]}: {cnt[k]:,}건 ({sh[k] * 100:.1f}%)")

    lines += ["", "## 기준 후보별 클래스 비율 (%, 여유/보통/혼잡) - 전체 기간 기준, 참고용"]
    lines += threshold_candidate_lines(df["occ"])
    if n_t >= 2:
        lines.append("> 경계를 정할 때는 이 표가 아니라 실행 후 저장되는 threshold_candidates_train.md(학습 구간만)를 보세요. "
                     "이 표에는 테스트 구간이 섞여 있습니다.")

    size = pd.cut(df["capacity"], [0, 30, 100, 300, 1e9], labels=["30면 이하", "31~100", "101~300", "300 초과"])
    g = df.groupby(size, observed=True)["occ"].agg(["count", "mean", "median"]).round(3)
    lines += ["", "## 규모별 점유율", "| 규모 | 관측 | 평균 | 중앙값 |", "|---|---|---|---|"]
    for k, r in g.iterrows():
        lines.append(f"| {k} | {int(r['count'])} | {r['mean']:.3f} | {r['median']:.3f} |")

    if n_t < 2:
        lines += ["", "> **시점이 1개뿐인 스냅샷입니다.** 30분 뒤 예측과 시간 순서 분할은 시계열 수집 후에 가능합니다."]
    os.makedirs(outdir, exist_ok=True)
    with open(os.path.join(outdir, "eda_summary.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    print("\n".join(lines))
    plot_occ_hist(df["occ"].to_numpy(), thr, os.path.join(outdir, "occupancy_hist.png"), n_t, n_lot)

## 4. 학습용 표 만들기 (시간 격자 위에서 병합)

In [ ]:
def build_dataset(df, interval_min, horizon_min, thr, holidays) -> tuple[pd.DataFrame, list[int]]:
    if horizon_min % interval_min:
        raise ValueError(f"예측 시점({horizon_min}분)이 수집 간격({interval_min}분)의 정수배여야 합니다.")
    lags = [m for m in (10, 30, 60) if m % interval_min == 0]

    def attach(base, offset_min, col):
        s = df[["lot_id", "t", "occ"]].copy()
        s["t"] = s["t"] - pd.Timedelta(minutes=offset_min)  # t+offset 의 값을 t 행에 붙인다
        return base.merge(s.rename(columns={"occ": col}), on=["lot_id", "t"], how="left")

    out = df[["lot_id", "t", "capacity", "occ"]].copy()
    out = attach(out, horizon_min, "occ_future")
    for L in lags:
        out = attach(out, -L, f"occ_lag{L}")
    out = out.dropna().reset_index(drop=True)
    if out.empty:
        raise ValueError(
            "30분 뒤 값과 과거 값을 모두 가진 행이 없습니다. 시점이 충분한 시계열 데이터가 필요합니다 "
            f"(현재 고유 시점 {df['t'].nunique()}개).")

    for L in lags:
        out[f"d{L}"] = out["occ"] - out[f"occ_lag{L}"]
    mins = out["t"].dt.hour * 60 + out["t"].dt.minute
    out["tod_sin"], out["tod_cos"] = np.sin(2 * np.pi * mins / 1440), np.cos(2 * np.pi * mins / 1440)
    dow = out["t"].dt.dayofweek
    out["dow_sin"], out["dow_cos"] = np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7)
    hol = out["t"].dt.normalize().isin(pd.to_datetime(holidays)) if holidays else False
    out["is_off"] = ((dow >= 5) | hol).astype(int)
    out["hour"] = out["t"].dt.hour
    out["log_cap"] = np.log(out["capacity"])
    out["y_now"] = to_class(out["occ"], thr)
    out["y_true"] = to_class(out["occ_future"], thr)
    return out, lags


def time_split(data, horizon_min, test_frac, val_frac):
    """시간 순서 분할. 학습 행의 '30분 뒤 시각'이 검증/테스트 구간으로 넘어가지 않게 간격을 둔다."""
    h = pd.Timedelta(minutes=horizon_min)
    span_days = (data["t"].max() - data["t"].min()).total_seconds() / 86400
    too_short = (f"기간이 너무 짧아 학습/검증/테스트를 나눌 수 없습니다(유효 데이터 약 {span_days:.1f}일). "
                 "날짜 단위로 구간을 자르므로 최소 2주 이상 수집하세요.")
    ts = np.sort(data["t"].unique())
    test_start = pd.Timestamp(ts[min(int(len(ts) * (1 - test_frac)), len(ts) - 1)]).normalize()
    train_full = data[data["t"] + h < test_start]
    test = data[data["t"] >= test_start]
    if train_full.empty or test.empty:  # (수정) 하루 안팎의 데이터면 아래 인덱싱에서 IndexError 가 났다
        raise ValueError(too_short)
    ts2 = np.sort(train_full["t"].unique())
    val_start = pd.Timestamp(ts2[min(int(len(ts2) * (1 - val_frac)), len(ts2) - 1)]).normalize()
    inner = train_full[train_full["t"] + h < val_start]
    val = train_full[train_full["t"] >= val_start]
    if min(len(inner), len(val), len(test)) == 0:
        raise ValueError(too_short)
    return inner, val, train_full, test, test_start, val_start


def fit_lot_stats(train: pd.DataFrame) -> dict:
    """주차장별 평균 점유율(학습 구간에서만 계산해 누수를 막는다)."""
    return {"lot": train.groupby("lot_id")["occ"].mean(),
            "slot": train.groupby(["lot_id", "is_off", "hour"])["occ"].mean(),
            "global": float(train["occ"].mean())}


def feature_matrix(df: pd.DataFrame, lags: list[int], st: dict) -> np.ndarray:
    lot_mean = df["lot_id"].map(st["lot"]).fillna(st["global"])
    idx = pd.MultiIndex.from_frame(df[["lot_id", "is_off", "hour"]])
    slot_mean = pd.Series(st["slot"].reindex(idx).to_numpy(), index=df.index).fillna(lot_mean)
    cols = ["occ"] + [f"d{L}" for L in lags] + ["tod_sin", "tod_cos", "dow_sin", "dow_cos", "is_off", "log_cap"]
    X = df[cols].copy()
    X["lot_mean"], X["lot_slot_mean"] = lot_mean, slot_mean
    return X.to_numpy(dtype=float)

## 5. 모델, 평가

In [ ]:
def model_specs(class_weight, fast: bool) -> dict:
    cs_lin = [0.1, 1] if fast else [0.01, 0.1, 1, 10]
    return {
        "LinearSVC": dict(kind="clf", grid=[{"C": c} for c in cs_lin],
                          build=lambda p: make_pipeline(StandardScaler(), LinearSVC(C=p["C"], class_weight=class_weight, max_iter=20000))),
        "RBF SVC": dict(kind="clf",
                        grid=[{"C": c, "gamma": g} for c in ([1] if fast else [1, 10]) for g in (["scale"] if fast else ["scale", 0.02, 0.2])],
                        build=lambda p: make_pipeline(StandardScaler(), SVC(kernel="rbf", C=p["C"], gamma=p["gamma"], class_weight=class_weight, cache_size=1000))),
        "SVR→구간화": dict(kind="reg",
                         grid=[{"C": c, "gamma": g} for c in ([1] if fast else [1, 10]) for g in (["scale"] if fast else ["scale", 0.02])],
                         build=lambda p: make_pipeline(StandardScaler(), SVR(kernel="rbf", C=p["C"], gamma=p["gamma"], epsilon=0.02, cache_size=1000))),
    }


def predict_class(model, kind, X, thr) -> tuple[np.ndarray, np.ndarray | None]:
    if kind == "clf":
        return model.predict(X).astype(int), None
    occ_hat = model.predict(X)
    return to_class(occ_hat, thr), occ_hat


def class_metrics(y_true, y_pred, y_now) -> dict:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    support = cm.sum(axis=1)
    recall = np.where(support > 0, np.diag(cm) / np.maximum(support, 1), np.nan)
    f1 = f1_score(y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    trans = y_true != y_now  # 30분 사이에 구간이 바뀐 행
    return {
        "macro_f1": float(f1[support > 0].mean()),
        "accuracy": float((y_true == y_pred).mean()),
        **{f"recall_{CLASS_NAMES[k]}": float(recall[k]) for k in range(3)},
        "acc_전환구간": float((y_pred[trans] == y_true[trans]).mean()) if trans.any() else float("nan"),
        "acc_유지구간": float((y_pred[~trans] == y_true[~trans]).mean()) if (~trans).any() else float("nan"),
        "n_전환": int(trans.sum()),
        "cm": cm,
    }


def choose(n: int, cap: int, rng) -> np.ndarray:
    """n개 중 cap개를 무작위로 고른 위치(오름차순). cap이 0이거나 n이 작으면 전부."""
    return np.sort(rng.choice(n, cap, replace=False)) if cap and n > cap else np.arange(n)


def md_table(df: pd.DataFrame, fmt="{:.3f}") -> str:
    head = "| " + " | ".join(df.columns) + " |\n|" + "---|" * len(df.columns) + "\n"
    body = ""
    for _, r in df.iterrows():
        body += "| " + " | ".join(fmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in r) + " |\n"
    return head + body

## 6. 그림 함수

In [ ]:
_FONT_WARNED = False


def setup_plot():
    global _FONT_WARNED
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
    names = {f.name for f in font_manager.fontManager.ttflist}
    for cand in ("Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR"):
        if cand in names:
            plt.rcParams["font.family"] = cand
            break
    else:
        if not _FONT_WARNED:
            _FONT_WARNED = True
            print("[경고] 한글 폰트를 찾지 못했습니다. 그래프의 한글이 네모(□)로 보일 수 있습니다. "
                  "(Colab: '!apt-get -qq install fonts-nanum' 실행 후 런타임 다시 시작)")
    plt.rcParams["axes.unicode_minus"] = False
    return plt


def style_axes(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
    ax.tick_params(colors=INK2, labelsize=9)
    ax.yaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def plot_occ_hist(occ, thr, path, n_t, n_lot):
    plt = setup_plot()
    fig, ax = plt.subplots(figsize=(7.5, 4.2), dpi=160)
    style_axes(ax)
    ax.hist(np.clip(occ, 0, 1), bins=np.linspace(0, 1, 21), color=BLUE, edgecolor="white", linewidth=1.5)
    top = ax.get_ylim()[1] * 1.18
    ax.set_ylim(0, top)
    sh = share_table(occ, thr)
    edges = [0, thr[0], thr[1], 1]
    for k in range(3):
        ax.text((edges[k] + edges[k + 1]) / 2, top * 0.95, f"{CLASS_NAMES[k]} {sh[k] * 100:.1f}%",
                ha="center", va="top", fontsize=10, color=INK)
    for v in thr:
        ax.axvline(v, color=INK2, linestyle=(0, (4, 3)), linewidth=1)
    ax.set_xlim(0, 1)
    ax.set_xlabel("점유율 (현재 주차 대수 ÷ 총 주차면)", color=INK2, fontsize=9)
    ax.set_ylabel("관측 수", color=INK2, fontsize=9)
    ax.set_title(f"점유율 분포와 혼잡도 구간 (관측 {len(occ):,}건 · 주차장 {n_lot}개 · 시점 {n_t}개)",
                 loc="left", fontsize=11, color=INK)
    fig.tight_layout()
    fig.savefig(path, facecolor="white")
    plt.show()  # 노트북에서는 여기서 화면에 표시된다
    plt.close(fig)


def plot_confusions(results: dict, path: str, title_suffix: str = ""):
    from matplotlib.colors import LinearSegmentedColormap
    plt = setup_plot()
    cmap = LinearSegmentedColormap.from_list("blue_seq", BLUE_SEQ)
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(3.4 * n, 4.2), dpi=160)
    for ax, (name, r) in zip(np.atleast_1d(axes), results.items()):
        cm = r["cm"]
        rate = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        ax.imshow(rate, cmap=cmap, vmin=0, vmax=1)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{cm[i, j]:,}\n{rate[i, j] * 100:.0f}%", ha="center", va="center", fontsize=8.5,
                        color="white" if rate[i, j] > 0.55 else INK)
        ax.set_xticks(range(3), CLASS_NAMES, fontsize=9, color=INK2)
        ax.set_yticks(range(3), CLASS_NAMES, fontsize=9, color=INK2)
        ax.set_xticks(np.arange(-.5, 3, 1), minor=True)
        ax.set_yticks(np.arange(-.5, 3, 1), minor=True)
        ax.grid(which="minor", color="white", linewidth=2)
        ax.tick_params(which="both", length=0)
        for s in ax.spines.values():
            s.set_visible(False)
        ax.set_xlabel("예측", fontsize=9, color=INK2)
        ax.set_ylabel("실제", fontsize=9, color=INK2)
        ax.set_title(f"{name}\nmacro F1 {r['macro_f1']:.3f}", fontsize=10, color=INK, loc="left")
    fig.text(0.01, 0.01, "칸 안: 건수 / 실제 클래스 대비 비율(재현율). 색이 진할수록 비율이 높음." + title_suffix,
             fontsize=8, color=INK2)
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(path, facecolor="white")
    plt.show()
    plt.close(fig)

## 7. 실행

여기부터는 위 함수들을 순서대로 호출합니다. 위에서 아래로 실행하세요.

### 7-1. 설정 확인

In [ ]:
if isinstance(DATA, (str, os.PathLike)):  # DATA = "경로" 처럼 문자열로 적어도 되게
    DATA = [DATA]
if not 0 < THR[0] < THR[1] <= 1.5:
    raise ValueError("THR 는 (낮은값, 높은값) 순서의 점유율 비율이어야 합니다 (예: (0.50, 0.85), 퍼센트 아님).")
if not DEMO and not DATA:
    raise ValueError("DEMO = False 이면 DATA 에 CSV 파일 또는 폴더 경로를 적어야 합니다.")

thr = tuple(THR)
rng = np.random.default_rng(SEED)
os.makedirs(OUTDIR, exist_ok=True)
print("설정 OK:", "합성 데이터(DEMO)" if DEMO else f"실제 데이터 {DATA}", "| 혼잡도 경계", thr, "| 결과 폴더", OUTDIR)

### 7-2. 데이터 불러오기

In [ ]:
if DEMO:
    print("=" * 70 + "\n[DEMO] 합성 데이터입니다. 아래 수치는 실제 주차장 성능이 아닙니다.\n" + "=" * 70)
    df = make_demo(interval_min=INTERVAL_MIN, seed=SEED)
else:
    try:
        df = load_snapshots(DATA, INTERVAL_MIN, DROP_CONSTANT_LOTS)
    except (FileNotFoundError, ValueError) as e:
        raise RuntimeError(f"[중단] {e}") from None
df.head()

### 7-3. 분포 확인(EDA)

스냅샷 한 장짜리 CSV라면 **이 셀까지만** 실행하면 됩니다.

In [ ]:
eda(df, thr, OUTDIR)

### 7-4. 학습용 표 만들기 + 시간 순서 분할

In [ ]:
try:
    data, lags = build_dataset(df, INTERVAL_MIN, HORIZON_MIN, thr, HOLIDAYS)
    inner, val, train_full, test, test_start, val_start = time_split(data, HORIZON_MIN, TEST_FRAC, VAL_FRAC)
except ValueError as e:
    raise RuntimeError(f"[중단] {e}\n분포 확인(EDA) 결과는 {os.path.abspath(OUTDIR)} 에 저장되어 있습니다.") from None

# (추가) 내부 학습 구간에 클래스가 하나뿐이면 SVM 이 학습 단계에서 예외를 던진다
if inner["y_true"].nunique() < 2:
    raise RuntimeError("[중단] 내부 학습 구간에 혼잡도 클래스가 1개뿐이라 SVM 을 학습할 수 없습니다. "
                       "수집 기간을 늘리거나 THR 경계를 조정하세요.")

days = (data["t"].max() - data["t"].min()).days + 1
if days < 14:
    print(f"[경고] 데이터 기간이 {days}일입니다. 요일 패턴을 보려면 2주 이상을 권장합니다.")
print(f"\n[split] 학습 {len(train_full):,} (내부 학습 {len(inner):,} + 검증 {len(val):,}) / 테스트 {len(test):,}")
print(f"        검증 시작 {val_start.date()}, 테스트 시작 {test_start.date()} "
      f"(테스트 {test['t'].dt.normalize().nunique()}일, 주말·휴일 {test.loc[test['is_off'] == 1, 't'].dt.normalize().nunique()}일)")

split_share = {nm: share_table(d["occ_future"], thr) for nm, d in
               [("학습", train_full), ("검증", val), ("테스트", test)]}
for nm, s in split_share.items():
    print(f"        {nm} 클래스 비율(여유/보통/혼잡): " + " / ".join(f"{x * 100:.1f}%" for x in s))
if min(split_share["학습"]) < 0.05:
    print("[경고] 학습 구간에서 5% 미만인 클래스가 있습니다. THR 로 경계 조정을 검토하세요(학습 구간 분포만 보고 정할 것).")

# 경계는 학습 구간 분포만 보고 정한다: 후보별 비율을 학습 구간(30분 뒤 점유율)만으로 따로 저장
cand = ["## 학습 구간 기준 후보별 클래스 비율 (%, 여유/보통/혼잡; 30분 뒤 점유율, 학습 구간만)",
        *threshold_candidate_lines(train_full["occ_future"]),
        "", "경계를 바꿔 다시 실행해도 시간 분할 지점은 그대로이므로, 이 표를 보고 THR 를 정해도 테스트 구간은 보지 않은 셈입니다."]
with open(os.path.join(OUTDIR, "threshold_candidates_train.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(cand) + "\n")
print("\n".join(cand[:5]))
if not HOLIDAYS:
    print("[알림] HOLIDAYS 를 지정하지 않았습니다. 평일에 낀 공휴일은 평일로 취급됩니다 "
          "(수집 기간에 공휴일이 있으면 'YYYY-MM-DD' 로 넘기세요).")

### 7-5. 특징 행렬 + 기준선

In [ ]:
# 검증용(내부 학습 통계) / 최종용(학습 전체 통계)
st_in, st_full = fit_lot_stats(inner), fit_lot_stats(train_full)
X_in, X_val = feature_matrix(inner, lags, st_in), feature_matrix(val, lags, st_in)
X_tr, X_te = feature_matrix(train_full, lags, st_full), feature_matrix(test, lags, st_full)
y_te, now_te = test["y_true"].to_numpy(), test["y_now"].to_numpy()

results: dict[str, dict] = {}
preds = {"y_true": y_te}

maj = int(np.bincount(train_full["y_true"], minlength=3).argmax())
results["기준선: 최빈 클래스"] = class_metrics(y_te, np.full_like(y_te, maj), now_te)
results["기준선: 지금 그대로"] = class_metrics(y_te, now_te, now_te)
preds["기준선: 최빈 클래스"], preds["기준선: 지금 그대로"] = np.full_like(y_te, maj), now_te
print("기준선 macro F1:", {k: round(v["macro_f1"], 3) for k, v in results.items()})

### 7-6. SVM 계열 모델 학습 (하이퍼파라미터 선택 → 테스트)

이 셀이 가장 오래 걸립니다. `FAST = False`이고 표본이 많으면 수 분~수십 분이 걸릴 수 있습니다. 진행 상황은 `[tune]` 줄로 출력됩니다.

In [ ]:
class_weight = None if NO_BALANCE else "balanced"
reg_rows = [{"모델": "지금 그대로(점유율 그대로)", "MAE": mean_absolute_error(test["occ_future"], test["occ"]),
             "RMSE": mean_squared_error(test["occ_future"], test["occ"]) ** 0.5}]
chosen = {}

# 모델 간 비교가 표본 차이에 흔들리지 않도록, 모든 SVM이 같은 학습·검증 표본을 쓴다
sel_in, sel_val, sel_tr = (choose(len(inner), MAX_TRAIN, rng), choose(len(val), MAX_VAL, rng),
                           choose(len(train_full), MAX_TRAIN, rng))
yv_cls, now_v = val["y_true"].to_numpy()[sel_val], val["y_now"].to_numpy()[sel_val]
Xv = X_val[sel_val]

for name, spec in model_specs(class_weight, FAST).items():
    kind = spec["kind"]
    ycol = "occ_future" if kind == "reg" else "y_true"
    Xa, ya = X_in[sel_in], inner[ycol].to_numpy()[sel_in]
    best, best_f1 = None, -1.0
    for p in spec["grid"]:
        t0 = time.time()
        m = spec["build"](p).fit(Xa, ya)
        f1 = class_metrics(yv_cls, predict_class(m, kind, Xv, thr)[0], now_v)["macro_f1"]
        print(f"[tune] {name} {p} 검증 macro F1 {f1:.4f} ({time.time() - t0:.0f}s)")
        if f1 > best_f1:
            best, best_f1 = p, f1
    chosen[name] = {"params": best, "val_macro_f1": best_f1}
    Xb, yb = X_tr[sel_tr], train_full[ycol].to_numpy()[sel_tr]
    model = spec["build"](best).fit(Xb, yb)
    pred, occ_hat = predict_class(model, kind, X_te, thr)
    results[name] = class_metrics(y_te, pred, now_te)
    preds[name] = pred
    if occ_hat is not None:
        preds["SVR 점유율 예측"] = occ_hat
        reg_rows.append({"모델": "SVR", "MAE": mean_absolute_error(test["occ_future"], occ_hat),
                         "RMSE": mean_squared_error(test["occ_future"], occ_hat) ** 0.5})
    print(f"[test] {name} {best} macro F1 {results[name]['macro_f1']:.4f}")

### 7-7. 결과 저장

In [ ]:
cols = ["macro_f1", "accuracy"] + [f"recall_{c}" for c in CLASS_NAMES] + ["acc_전환구간", "acc_유지구간"]
table = pd.DataFrame({k: {c: v[c] for c in cols} for k, v in results.items()}).T
table.index.name = "모델"
table.to_csv(os.path.join(OUTDIR, "metrics.csv"), encoding="utf-8-sig")
for k, v in results.items():
    pd.DataFrame(v["cm"], index=[f"실제_{c}" for c in CLASS_NAMES], columns=[f"예측_{c}" for c in CLASS_NAMES]) \
        .to_csv(os.path.join(OUTDIR, f"confusion_{k.replace(':', '').replace(' ', '_').replace('→', '_')}.csv"),
                encoding="utf-8-sig")
out_pred = test[["lot_id", "t", "occ", "occ_future", "y_now"]].reset_index(drop=True).copy()
for k, v in preds.items():
    out_pred[k] = v
out_pred.to_csv(os.path.join(OUTDIR, "test_predictions.csv"), index=False, encoding="utf-8-sig")

reg = pd.DataFrame(reg_rows)
reg.to_csv(os.path.join(OUTDIR, "regression_metrics.csv"), index=False, encoding="utf-8-sig")
cfg = {"demo": DEMO, "thresholds": thr, "interval_min": INTERVAL_MIN, "horizon_min": HORIZON_MIN,
       "lags_min": lags, "class_weight": class_weight, "max_train": MAX_TRAIN, "seed": SEED,
       "n_rows": {"train_full": len(train_full), "inner": len(inner), "val": len(val), "test": len(test)},
       "val_start": str(val_start.date()), "test_start": str(test_start.date()),
       "class_share_by_split(여유/보통/혼잡)": split_share, "chosen": chosen}
with open(os.path.join(OUTDIR, "run_config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2, default=str)
with open(os.path.join(OUTDIR, "summary.md"), "w", encoding="utf-8") as f:
    if DEMO:
        f.write("> **합성 데이터로 코드만 점검한 결과입니다. 실제 성능이 아닙니다.**\n\n")
    f.write("## 테스트 구간 분류 성능\n\n" + md_table(table.reset_index()) + "\n")
    f.write("\n## 점유율 회귀 오차\n\n" + md_table(reg) + "\n")
    f.write("\n## 선택된 하이퍼파라미터(검증 macro F1 기준)\n\n"
            + "\n".join(f"- {k}: {v['params']} (검증 macro F1 {v['val_macro_f1']:.3f})" for k, v in chosen.items()) + "\n")
print(f"결과 저장: {os.path.abspath(OUTDIR)}")

### 7-8. 결과 확인

In [ ]:
if DEMO:
    print("※ 합성 데이터 결과입니다. 실제 성능이 아닙니다.\n")
print("테스트 구간 분류 성능")
display(table.round(3))

In [ ]:
print("점유율 회귀 오차 (낮을수록 좋음)")
display(reg.round(4))
print("선택된 하이퍼파라미터(검증 macro F1 기준)")
for k, v in chosen.items():
    print(f"- {k}: {v['params']} (검증 macro F1 {v['val_macro_f1']:.3f})")

In [ ]:
plot_confusions(results, os.path.join(OUTDIR, "confusion_matrices.png"), "  [합성 데이터]" if DEMO else "")